# Notebook 06 — SMAP MAML-AE (Corrected Core)

**Why this notebook exists.** The original meta-training loop adapted a `copy.deepcopy`
of the model, computed the query loss on that copy, then called `.backward()` / `.step()`
on the *original* model. The deepcopy severs the graph, so the meta-model received **no
gradients and never trained** — every "MAML-AE" result came from the untrained
initialization. This notebook uses a corrected FOMAML core (query-loss gradients are
computed on the adapted learner and **transferred onto the meta-model**), and compares
it against a **properly-trained** static AE under an identical few-shot protocol.

Run order: (1) meta-train MAML, (2) train static baseline, (3) evaluate both. Training
checkpoints and resumes, so an interrupted Kaggle session loses nothing.


### ▶ Run instructions
1. First produce `channel_data_normalized.pkl` if you don't already have it: run
   `02_preprocessing.ipynb` against the raw SMAP/MSL data (adjust `TRAIN_PATH`/`TEST_PATH`/
   `LABEL_PATH` in its first code cell to wherever your `data/raw/train/`, `data/raw/test/`,
   `labeled_anomalies.csv` live). That notebook's later cells save `channel_data_normalized.pkl`
   to `data/processed/`.
2. Create a Kaggle Dataset containing `channel_data_normalized.pkl`, `task_splits_25feat.json`,
   and `config.json`, and attach it as input. Set **accelerator to GPU**.
3. Use **Save Version → "Save & Run All (Commit)"**. 30k-step meta-training is expected to take
   roughly the same order as SWaT's (1-2h on GPU) — SMAP's windows are smaller (25 features vs
   SWaT's 51), so this should not run longer.
4. If ever interrupted, attach this notebook's own previous output as an additional input and
   Commit again — startup auto-resumes from the newest checkpoint found in either
   `/kaggle/working` or any attached input dataset.

## 1 — Imports, paths, data

In [ ]:
import os, json, pickle, copy, time, warnings
import numpy as np
import torch, torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

OUTPUT_PATH = "/kaggle/working"
WORK_MODELS = f"{OUTPUT_PATH}/models"
os.makedirs(WORK_MODELS, exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/results", exist_ok=True)

def _find(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None

with open(_find("channel_data_normalized.pkl"), "rb") as f:
    channel_data = pickle.load(f)
with open(_find("task_splits_25feat.json"), "r") as f:
    task_splits_25 = json.load(f)

meta_train_tasks = task_splits_25['meta_train']
meta_val_tasks   = task_splits_25['meta_val']
meta_test_25     = task_splits_25['meta_test']
# Exclude P-4 (sensor dropout) and D-12 (insufficient windows) — as in original eval
EVAL_CHANNELS = [ch for ch in meta_test_25 if ch not in ['P-4','D-12']]
print(f"meta-train {len(meta_train_tasks)} | meta-val {len(meta_val_tasks)} | eval {EVAL_CHANNELS}")

def _all_dirs():
    d = [WORK_MODELS]
    for r, _, _ in os.walk("/kaggle/input"):
        d.append(r)
    return d

def resolve_ckpt(name):
    best_p, best_s = None, -1
    for d in _all_dirs():
        p = os.path.join(d, name)
        if os.path.exists(p):
            try:
                s = torch.load(p, map_location="cpu").get("step", 0)
            except Exception:
                s = 0
            if s >= best_s:
                best_s, best_p = s, p
    return best_p, best_s

## 2 — Model classes (verbatim from Notebook 4/5)

In [ ]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_size=25, hidden1=64, hidden2=32, latent=16):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden1, batch_first=True)
        self.lstm2 = nn.LSTM(hidden1,   hidden2, batch_first=True)
        self.fc    = nn.Linear(hidden2, latent)
    def forward(self, x):
        out, _      = self.lstm1(x)
        _, (h_n, _) = self.lstm2(out)
        return self.fc(h_n.squeeze(0))

class LSTMDecoder(nn.Module):
    def __init__(self, latent=16, hidden1=32, hidden2=64, output_size=25, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.lstm1   = nn.LSTM(latent,  hidden1, batch_first=True)
        self.lstm2   = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.fc      = nn.Linear(hidden2, output_size)
    def forward(self, z):
        z_rep  = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.lstm1(z_rep)
        out, _ = self.lstm2(out)
        return self.fc(out)

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=25, seq_len=30, hidden1=64, hidden2=32, latent=16):
        super().__init__()
        self.encoder = LSTMEncoder(input_size, hidden1, hidden2, latent)
        self.decoder = LSTMDecoder(latent, hidden2, hidden1, input_size, seq_len)
    def forward(self, x): return self.decoder(self.encoder(x))
    def reconstruction_error(self, x):
        x_hat = self.forward(x)
        return torch.mean((x - x_hat) ** 2, dim=(1, 2))

class MLPAutoencoder(nn.Module):
    def __init__(self, input_size=25, seq_len=30, latent=16):
        super().__init__()
        self.seq_len, self.input_size = seq_len, input_size
        flat = seq_len * input_size
        self.encoder = nn.Sequential(nn.Linear(flat,256), nn.ReLU(),
                                     nn.Linear(256,64), nn.ReLU(), nn.Linear(64,latent))
        self.decoder = nn.Sequential(nn.Linear(latent,64), nn.ReLU(),
                                     nn.Linear(64,256), nn.ReLU(), nn.Linear(256,flat))
    def forward(self, x):
        b = x.shape[0]
        z = self.encoder(x.reshape(b,-1))
        return self.decoder(z).reshape(b, self.seq_len, self.input_size)
    def reconstruction_error(self, x):
        x_hat = self.forward(x)
        return torch.mean((x - x_hat) ** 2, dim=(1, 2))
print("model classes ready")


## 3 — Corrected FOMAML core

**THE FIX** is in `outer_step`: gradients are computed on the *adapted learner* with
`torch.autograd.grad`, then written onto the meta-model's `.grad` before the optimizer
steps. The old code let those gradients die on the discarded deepcopy.

In [ ]:
criterion = nn.MSELoss()

def sample_episode(channel_id, support_size=20, query_size=20, rng=np.random):
    w = channel_data[channel_id]['normal_windows']           # normal-only meta-training
    idx = rng.permutation(len(w)); need = support_size + query_size
    if len(w) < need:
        s = w[rng.choice(len(w), support_size, replace=True)]
        q = w[rng.choice(len(w), query_size, replace=True)]
    else:
        s = w[idx[:support_size]]; q = w[idx[support_size:need]]
    return (torch.tensor(s, dtype=torch.float32).to(DEVICE),
            torch.tensor(q, dtype=torch.float32).to(DEVICE))

def inner_adapt(model, support, inner_lr=0.01, inner_steps=10):
    learner = copy.deepcopy(model); learner.train()
    opt = torch.optim.SGD(learner.parameters(), lr=inner_lr)
    for _ in range(inner_steps):
        opt.zero_grad(); loss = criterion(learner(support), support)
        loss.backward(); opt.step()
    return learner

def outer_step(model, outer_opt, task_batch, inner_lr=0.01, inner_steps=10,
               support_size=20, query_size=20, train=True, rng=np.random):
    n_params = len(list(model.parameters())); accum=[None]*n_params; meta_loss=0.0
    for ch in task_batch:
        support, query = sample_episode(ch, support_size, query_size, rng)
        learner = inner_adapt(model, support, inner_lr, inner_steps)
        qloss = criterion(learner(query), query)
        if train:
            g = torch.autograd.grad(qloss, learner.parameters())          # grads wrt ADAPTED params
            accum = [gi.detach() if a is None else a+gi.detach() for a,gi in zip(accum,g)]
        meta_loss += qloss.item()
    meta_loss /= len(task_batch)
    if train:
        outer_opt.zero_grad()
        for p,a in zip(model.parameters(), accum):
            p.grad = a/len(task_batch)                                     # TRANSFER onto meta-model
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        outer_opt.step()
    return meta_loss
print("corrected FOMAML core ready")


## 4 — Meta-train MAML (faithful hyperparameters, checkpoint/resume)

In [ ]:
INNER_LR=0.01; OUTER_LR=0.001; INNER_STEPS=10; TPB=4; SUP=20; QRY=20
N_OUTER=30000; VAL_EVERY=500
CKPT_NAME = "smap_maml_ckpt.pt"; BEST_NAME = "smap_maml_best.pt"
WORK_CKPT = f"{WORK_MODELS}/{CKPT_NAME}"; WORK_BEST = f"{WORK_MODELS}/{BEST_NAME}"
rng = np.random.RandomState(42)

model = LSTMAutoencoder(25).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=OUTER_LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=5, min_lr=1e-5)
start=0; best=float('inf'); hist={'step':[],'train':[],'val':[]}
rp, rs = resolve_ckpt(CKPT_NAME)
if rp is not None and rs > 0:
    ck=torch.load(rp, map_location=DEVICE); model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt']); start=ck['step']; best=ck['best']; hist=ck['hist']
    print("resumed @", start, "from", rp)
else:
    print("training from scratch")

def meta_val():
    return float(np.mean([outer_step(model,opt,[c],INNER_LR,INNER_STEPS,SUP,QRY,train=False,rng=rng)
                          for c in meta_val_tasks]))

t0=time.time()
for step in range(start+1, N_OUTER+1):
    batch = rng.choice(meta_train_tasks, size=TPB, replace=False).tolist()
    model.train(); tl=outer_step(model,opt,batch,INNER_LR,INNER_STEPS,SUP,QRY,train=True,rng=rng)
    if step % VAL_EVERY == 0:
        vl=meta_val(); sched.step(vl)
        hist['step'].append(step); hist['train'].append(tl); hist['val'].append(vl)
        print(f"step {step:6d} | train {tl:.6f} | val {vl:.6f} | {time.time()-t0:.0f}s")
        torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'step':step,
                    'best':best,'hist':hist}, WORK_CKPT)
        if vl < best:
            best=vl; torch.save({'model_state_dict':model.state_dict(),'val_loss':vl,'step':step}, WORK_BEST)
            print(f"   best saved (val {vl:.6f})")
print("meta-training done. best val", best)
json.dump(hist, open(f"{OUTPUT_PATH}/results/smap_maml_history.json","w"), indent=2)


## 5 — Properly-trained static AE baseline

The honest bar. A conventional AE trained to convergence on pooled normal windows from
the meta-train channels — **not** a random init nudged a few steps. Same architecture,
same data, no meta-learning.

In [ ]:
SB=f"{OUTPUT_PATH}/models/smap_static_ae.pt"
pool = np.concatenate([channel_data[c]['normal_windows'] for c in meta_train_tasks], axis=0).astype(np.float32)
p = np.random.RandomState(42).permutation(len(pool)); cut=int(0.9*len(pool))
tr = torch.tensor(pool[p[:cut]]); va = torch.tensor(pool[p[cut:]])
print("static pool", pool.shape)

static = LSTMAutoencoder(25).to(DEVICE); sopt=torch.optim.Adam(static.parameters(),lr=1e-3)
BATCH=128; best_s=1e9; bad=0; best_state=None
for ep in range(150):
    static.train(); pi=torch.randperm(len(tr))
    for st in range(0,len(tr),BATCH):
        b=tr[pi[st:st+BATCH]].to(DEVICE); sopt.zero_grad()
        loss=criterion(static(b),b); loss.backward(); sopt.step()
    static.eval()
    with torch.no_grad(): vl=criterion(static(va.to(DEVICE)),va.to(DEVICE)).item()
    if vl<best_s-1e-6: best_s=vl; bad=0; best_state={k:v.clone() for k,v in static.state_dict().items()}
    else:
        bad+=1
        if bad>=15: break
    if (ep+1)%10==0: print(f"static ep{ep+1} val {vl:.6f}")
static.load_state_dict(best_state); torch.save({'model_state_dict':static.state_dict(),'val':best_s}, SB)
print("static baseline trained. best val", best_s)


## 6 — Evaluation protocol (verbatim helpers) + adaptation held equal

In [ ]:
def adapt_model(model, support, n_steps=5, lr=0.01):
    learner = copy.deepcopy(model); learner.train()
    o = torch.optim.SGD(learner.parameters(), lr=lr)
    for _ in range(n_steps):
        o.zero_grad(); l=criterion(learner(support), support); l.backward(); o.step()
    return learner

def compute_threshold(model, support, multiplier=2.0):
    model.eval()
    with torch.no_grad(): errors = model.reconstruction_error(support)
    m = errors.mean().item()
    return m + multiplier*errors.std().item() if len(errors)>1 else m*1.20

def compute_metrics(scores, labels, tau):
    preds=(scores>tau).astype(int)
    if 0<preds.sum()<len(preds):
        f1=f1_score(labels,preds,zero_division=0); pr=precision_score(labels,preds,zero_division=0)
        rc=recall_score(labels,preds,zero_division=0)
    else: f1=pr=rc=0.0
    au=roc_auc_score(labels,scores) if len(np.unique(labels))>1 else 0.5
    return {'f1':round(float(f1),4),'precision':round(float(pr),4),
            'recall':round(float(rc),4),'auroc':round(float(au),4)}

def build_eval_query(channel_id, k_shot, support_seed=42, max_normal_query=100):
    rng=np.random.RandomState(support_seed)
    nw=channel_data[channel_id]['normal_windows'].copy()
    aw=channel_data[channel_id]['anomaly_windows'].copy()
    rng.shuffle(nw); rng.shuffle(aw)
    support=nw[:k_shot]; rem=nw[k_shot:k_shot+max_normal_query]
    q=np.concatenate([aw,rem],0); y=np.concatenate([np.ones(len(aw)),np.zeros(len(rem))])
    idx=rng.permutation(len(q)); q,y=q[idx],y[idx]
    return (torch.tensor(support,dtype=torch.float32).to(DEVICE),
            torch.tensor(q,dtype=torch.float32).to(DEVICE), y)

# Load trained models
maml_model = LSTMAutoencoder(25).to(DEVICE)
maml_model.load_state_dict(torch.load(resolve_ckpt(BEST_NAME)[0], map_location=DEVICE)['model_state_dict'])
static_model = LSTMAutoencoder(25).to(DEVICE)
static_model.load_state_dict(torch.load(SB, map_location=DEVICE)['model_state_dict'])
ADAPT_STEPS = 5   # SAME test-time adaptation for MAML and Static -> isolates the meta-training effect
print("eval helpers + trained models ready")


## 7 — Per-channel evaluation (MAML vs properly-trained Static + baselines)

In [ ]:
def evaluate_channel(ch, k_shot, seed):
    support, query, labels = build_eval_query(ch, k_shot, support_seed=seed)
    out={}
    # MAML-AE: meta-trained theta*, adapt ADAPT_STEPS
    a=adapt_model(maml_model, support, n_steps=ADAPT_STEPS, lr=0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['MAML-AE']=compute_metrics(s, labels, compute_threshold(a, support))
    # Static-AE: properly pre-trained, SAME adaptation budget
    a=adapt_model(static_model, support, n_steps=ADAPT_STEPS, lr=0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['Static-AE']=compute_metrics(s, labels, compute_threshold(a, support))
    # MLP-AE from scratch (50 steps) — extra baseline
    a=adapt_model(MLPAutoencoder(25).to(DEVICE), support, n_steps=50, lr=0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['MLP-AE']=compute_metrics(s, labels, compute_threshold(a, support))
    # Isolation Forest
    sf=support.cpu().numpy().reshape(len(support),-1); qf=query.cpu().numpy().reshape(len(query),-1)
    iso=IsolationForest(n_estimators=100, contamination='auto', random_state=seed).fit(sf)
    isc=-iso.score_samples(qf); ip=(iso.predict(qf)==-1).astype(int)
    if 0<ip.sum()<len(ip):
        f1=f1_score(labels,ip,zero_division=0); pr=precision_score(labels,ip,zero_division=0); rc=recall_score(labels,ip,zero_division=0)
    else: f1=pr=rc=0.0
    au=roc_auc_score(labels,isc) if len(np.unique(labels))>1 else 0.5
    out['Isolation-Forest']={'f1':round(float(f1),4),'precision':round(float(pr),4),'recall':round(float(rc),4),'auroc':round(float(au),4)}
    return out

K_SHOTS=[1,5,10]; SEEDS=[42,123,456,789,1024]
METHODS=['MAML-AE','Static-AE','MLP-AE','Isolation-Forest']; METRICS=['f1','precision','recall','auroc']
all_results={}
for ch in EVAL_CHANNELS:
    all_results[ch]={}
    for k in K_SHOTS:
        store={m:{mt:[] for mt in METRICS} for m in METHODS}
        for seed in SEEDS:
            np.random.seed(seed); torch.manual_seed(seed)
            r=evaluate_channel(ch,k,seed)
            for m in METHODS:
                for mt in METRICS: store[m][mt].append(r[m][mt])
        all_results[ch][k]=store
    print(f"  {ch:6s} done")
json.dump(all_results, open(f"{OUTPUT_PATH}/results/smap_all_results.json","w"), indent=2)
print("evaluation complete")


## 8 — Macro results: the pivotal comparison

In [ ]:
print(f"{'shot':>4} | " + " | ".join(f"{m:>18s}" for m in METHODS))
for k in K_SHOTS:
    cells_=[]
    for m in METHODS:
        f1=np.mean([np.mean(all_results[ch][k][m]['f1'])    for ch in EVAL_CHANNELS])
        au=np.mean([np.mean(all_results[ch][k][m]['auroc']) for ch in EVAL_CHANNELS])
        cells_.append(f"F1 {f1:.3f}/AU {au:.3f}")
    print(f"{k:>4} | " + " | ".join(f"{c:>18s}" for c in cells_))

print("\nMAML-AE minus Static-AE (macro):")
for k in K_SHOTS:
    for mt in ['f1','auroc']:
        a=np.mean([np.mean(all_results[ch][k]['MAML-AE'][mt])  for ch in EVAL_CHANNELS])
        b=np.mean([np.mean(all_results[ch][k]['Static-AE'][mt]) for ch in EVAL_CHANNELS])
        print(f"  {k:2d}-shot {mt:5s}: {a-b:+.4f}")


## How to read this

- **MAML-AE > Static-AE** (esp. at 1-shot): meta-training helps; the earlier null was the
  no-op bug, and MAML genuinely adapts faster from few normal shots.
- **MAML-AE ≈ Static-AE** under this *correct* implementation: the honest-negative is
  **real** — now defensible, because both models actually trained and adaptation is held
  equal. This is a legitimate, publishable finding.
- The silhouette diagnostics predict limited headroom on structured single benchmarks, so
  a small or null gap here would not be surprising — but it would finally be a true result.
